In [ ]:
import os
import gc
import subprocess
import logging
from glob import glob
from os.path import join

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
import rioxarray as rxr
import xarray as xr
import zarr

import matplotlib.pyplot as plt

from shapely.geometry import mapping
import warnings
import pyproj

# Set the PROJ data directory for coordinate reference system operations (can be left out if crs works)
pyproj.datadir.set_data_dir("/opt/conda/envs/macrosystems/share/proj")

In [ ]:
def extract_gridmet_means(row, year, start_date, end_date, data_folder, pad=0.05):
    """
    Extract mean GRIDMET climate variables for a geometry over a date range.

    Inputs:
        row: pandas.Series or geopandas.GeoSeries
            Row containing a geometry column in EPSG:5070
        year: int
            Year corresponding to the GRIDMET NetCDF files
        start_date: string
            Start date for temporal subset selection
        end_date: string
            End date for temporal subset selection
        data_folder: string
            Base folder containing GRIDMET NetCDF files
        pad: float
            Amount to expand bounds if no grid cells are found (default=0.05)

    Returns:
        out: dict
            Dictionary containing mean values for each GRIDMET variable
    """

    # Mapping between GRIDMET file prefixes and NetCDF variable names
    VAR_MAP = {
        "pr": "precipitation_amount",
        "tmmn": "air_temperature",
        "tmmx": "air_temperature",
        "vpd": "mean_vapor_pressure_deficit",
    }

    # Convert geometry from EPSG:5070 to WGS84 for GRIDMET coordinate matching
    geom = gpd.GeoSeries(row.geometry, crs="EPSG:5070").to_crs(epsg=4326).iloc[0]
    minx, miny, maxx, maxy = geom.bounds

    out = {}

    for var, nc_var in VAR_MAP.items():

        # Open yearly GRIDMET dataset
        ds = xr.open_dataset(join(data_folder, "gridmet", f"{var}_{year}.nc"))

        # Spatial and temporal subset
        ds_sub = ds.sel(
            lon=slice(minx, maxx),
            lat=slice(maxy, miny),
            day=slice(start_date, end_date),
        )

        # Expand bounds slightly if no cells were selected
        if ds_sub.sizes.get("lat", 0) == 0 or ds_sub.sizes.get("lon", 0) == 0:
            ds_sub = ds.sel(
                lon=slice(minx - pad, maxx + pad),
                lat=slice(miny - pad, maxy + pad),
                day=slice(start_date, end_date),
            )

        # Compute mean value across time and space
        out[var] = float(ds_sub[nc_var].mean(("day", "lat", "lon")))

        # Convert temperatures from Kelvin to Celsius
        if var == "tmmn" or var == "tmmx":
            out[var] = out[var] - 273.15

        ds.close()

    return out


def mean_cbi(geom_gdf, year, cbi_da):
    """
    Calculate the mean CBI value within a geometry for a given year.

    Inputs:
        geom_gdf: geopandas.GeoDataFrame
            GeoDataFrame containing the geometry of interest
        year: int or string
            Year used to select the corresponding raster band
        cbi_da: xarray.DataArray
            Raster DataArray containing CBI values by band

    Returns:
        float
            Mean CBI value within the clipped geometry
    """

    # Convert year to raster band index
    band = int(year) - 2000 + 1

    # Select the appropriate band from the raster
    da = cbi_da.sel(band=band)
    
    # Get geometry bounds for an initial bounding box clip
    minx, miny, maxx, maxy = geom_gdf.geometry.bounds.iloc[0]
    da = da.rio.clip_box(minx, miny, maxx, maxy)
    
    # Clip raster to the exact geometry
    da = da.rio.clip(geom_gdf.geometry, geom_gdf.crs)
    
    return float(da.mean().values)


def cal_area(gdf):
    """
    Calculate the total geometry area in square kilometers.

    Inputs:
        gdf: geopandas.GeoDataFrame
            GeoDataFrame containing geometries

    Returns:
        float
            Total area of all geometries in square kilometers
    """

    # Sum geometry areas and convert from square meters to square kilometers
    return gdf.geometry.area.sum() / 1e6


def get_diff_overlay_geom(fired_gdf, fire1_id, fire2_id):
    """
    Compute intersection and difference geometries between two fire events.

    Inputs:
        fired_gdf: geopandas.GeoDataFrame
            GeoDataFrame containing fire geometries with a 'merge_id' column
        fire1_id: int or str
            ID of first fire event
        fire2_id: int or str
            ID of second fire event

    Returns:
        diff: geopandas.GeoSeries
            Geometry representing parts of fire2 not in fire1
        overlay: geopandas.GeoSeries
            Geometry representing intersection of fire1 and fire2
        fire1: geopandas.GeoDataFrame
            Subset of fired_gdf for fire1_id
        fire2: geopandas.GeoDataFrame
            Subset of fired_gdf for fire2_id
    """

    fire1 = fired_gdf.loc[fired_gdf.merge_id == fire1_id]
    fire2 = fired_gdf.loc[fired_gdf.merge_id == fire2_id]
    
    overlay = gpd.overlay(fire1, fire2, how="intersection").geometry
    diff = gpd.overlay(fire2, fire1, how="difference").geometry

    return diff, overlay, fire1, fire2


def check_forest_cover(year, geometry, data_folder):
    """
    Estimate forest cover percentage and dominant NLCD class within a geometry.

    Inputs:
        year: int
            Year of interest; if 2025, it is mapped back to 2024
        geometry: geopandas.GeoSeries or GeoDataFrame
            Geometry used for raster clipping (assumes single feature via .iloc[0])
        data_folder: string
            Base folder containing NLCD raster data

    Returns:
        dominant: int or None
            Most common NLCD land cover class within the geometry
        forest_perc: float
            Percentage of forested pixels (NLCD classes 41, 42, 43)
    """

    # NLCD does not have 2025 data; fallback to 2024
    if year == 2025:
        year = 2024

    # Find NLCD raster file for the given year
    nlcd_path = glob(join(data_folder, "nlcd", f"*{year}*.tif"))[0]

    # Extract single geometry from input series
    geom = geometry.iloc[0]

    # Clip raster to geometry extent
    area = rxr.open_rasterio(nlcd_path).rio.clip([geom], from_disk=True)
    masked = area.values

    # Remove background/invalid class (250)
    non_zero_values = masked[masked != 250]

    # NLCD forest classes
    forest_classes = [41, 42, 43]

    # Count forest pixels
    forest_pixel_count = sum(np.isin(non_zero_values, forest_classes))
    total_pixels = len(non_zero_values)

    # Compute forest percentage and dominant land cover class
    if total_pixels > 0:
        forest_perc = (forest_pixel_count / total_pixels) * 100
        dominant = pd.Series(non_zero_values.flatten()).mode().iloc[0]
    else:
        dominant = None
        forest_perc = 0

    return dominant, forest_perc


def get_ecoregion(ecoregions, geom):
    """
    Identify the ecoregion code overlapping a given geometry.

    Inputs:
        ecoregions: geopandas.GeoDataFrame
            Ecoregion polygons containing a 'US_L3CODE' attribute
        geom: geopandas.GeoDataFrame or GeoSeries
            Geometry to intersect with ecoregions

    Returns:
        str or int
            US_L3CODE of the first overlapping ecoregion, or "N/A" if no match/error
    """

    try:
        # Compute spatial intersection between geometry and ecoregions
        overlap = ecoregions.overlay(geom)

        # Return the first matching ecoregion code
        return overlap['US_L3CODE'].values[0]

    except:
        # Fallback if no overlap or overlay fails
        return "N/A"

def setup_logger(data_folder):
    """
    Configure and initialize a file-based logger for the pipeline.

    Inputs:
        data_folder: string
            Base directory where a 'logs' folder will be created

    Returns:
        logger: logging.Logger
            Configured logger instance writing to run.log
    """

    # Ensure logs directory exists
    os.makedirs(os.path.join(data_folder, "logs"), exist_ok=True)

    # Configure root logging to write to file with timestamped entries
    logging.basicConfig(
        filename=os.path.join(data_folder, "logs", "run.log"),
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s"
    )

    # Return logger instance for use in pipeline
    return logging.getLogger()

def padded_bounds(geom, da):
    """
    Expand geometry bounds by one raster pixel in all directions.

    Inputs:
        geom: shapely geometry
            Input geometry used to compute spatial bounds
        da: xarray.DataArray
            Raster DataArray used to infer pixel resolution (x and y spacing)

    Returns:
        tuple
            Padded bounding box as (minx, miny, maxx, maxy)
    """

    # Estimate raster resolution from coordinate spacing
    xres = abs(da.x[1] - da.x[0])
    yres = abs(da.y[1] - da.y[0])

    # Original geometry bounds
    minx, miny, maxx, maxy = geom.bounds

    # Expand bounds by one pixel in each direction
    return (
        minx - xres,
        miny - yres,
        maxx + xres,
        maxy + yres,
    )


def plot_cbi_row(row, cbi_da): 
    """
    Plot CBI values for a single geometry and year.

    Inputs:
        row: pandas.Series
            Row containing geometry and an 'ig_year_1' field
        cbi_da: xarray.DataArray
            CBI raster data (Zarr-backed), with a 'band' dimension

    Returns:
        None
            Displays a matplotlib plot of clipped CBI data with geometry overlay
    """

    # Extract geometry and corresponding year
    geom = row.geometry 
    year = int(row["ig_year_1"]) 
    band = year - 2000 + 1 

    # Select raster band for the target year
    da = cbi_da.sel(band=band) 

    # Bounding box prefilter (reduces data volume before precise clipping)
    minx, miny, maxx, maxy = padded_bounds(geom, da) 
    da = da.rio.clip_box(minx, miny, maxx, maxy) 

    # Convert geometry to GeoDataFrame for rioxarray clipping
    geom_gdf = gpd.GeoDataFrame(geometry=[geom], crs=cbi_da.rio.crs) 
    da = da.rio.clip(geom_gdf.geometry, geom_gdf.crs) 

    # Plot raster and overlay geometry boundary
    fig, ax = plt.subplots(figsize=(5, 5)) 
    da.plot(ax=ax, cmap="plasma", add_colorbar=True) 
    geom_gdf.boundary.plot(ax=ax, color="black", linewidth=2) 

    ax.set_title(f"CBI — {year}") 
    ax.set_axis_off() 

    plt.show()

In [ ]:
# Base directory containing all reburn analysis data
data_path = os.path.join("/home", "jovyan", "work", "reburn_data")

# CBI raster inputs (GeoTIFF and optional Zarr conversion target)
tif_path = os.path.join(data_path, "cbi_tif", "extendedwest_fired_bccbi.tif")
zarr_path = os.path.join(data_path, "cbi_zarr", "cbi.zarr")

# Ecoregion shapefile for western US Level III ecoregions
western_eco_path = join(
    data_path, "EPA-ecoregions", "western_us", "western_us_eco_l3.shp"
)

# Fire event datasets and outputs at different pipeline stages
fired_gdf_path = os.path.join(data_path, "fired_conus_ak_2000_to_2025_events_merged.gpkg")
clipped_path = os.path.join(data_path, "clipped_firedpy", "clipped_firedpy.gpkg")
overlay_path = os.path.join(data_path, "firedpy_overlay_full", "firedpy_overlay_full.gpkg")
final_path = os.path.join(data_path, "firedpy_severities_overlay", "firedpy_severities_overlay.gpkg")

# Initialize logger for pipeline tracking
logger1 = setup_logger(data_path)

# Open CBI raster as a chunked DataArray for efficient spatial processing
cbi_ds = rxr.open_rasterio(tif_path, chunks={"band": 1, "x": 512, "y": 512})

# Optional conversion of GeoTIFF to Zarr format for faster repeated access
make_zarr = False
if make_zarr:
    cbi_ds.to_zarr(zarr_path, mode="w")

# Path to full US EPA Level III ecoregions shapefile
full_eco_path = join(
    data_path, "EPA-ecoregions", "us_eco_l3", "us_eco_l3.shp"
)

# Load ecoregions into GeoDataFrame
full_eco = gpd.read_file(full_eco_path)

# Define subset of ecoregion codes corresponding to western forest regions
western_forests_regions = (
    [str(i) for i in list(range(27))] + 
    [str(i) for i in list(range(41, 44))] +
    [str(i) for i in list(range(77, 82))]
)

# Filter to western forest-related ecoregions and reproject to analysis CRS
western_epa = full_eco[full_eco['US_L3CODE'].isin(western_forests_regions)].to_crs(epsg=5070)

In [ ]:
# Load CBI dataset from Zarr store for faster repeated access
# Rename default variable name to "cbi" for clarity
cbi_da = (
    xr.open_zarr(zarr_path)
    .rename_vars({"__xarray_dataarray_variable__": "cbi"})["cbi"]
    .rio.set_spatial_dims(x_dim="x", y_dim="y")
    .rio.write_crs("EPSG:5070")
)

In [ ]:
# Load fire event GeoDataFrame from geopackage
fired_gdf = gpd.read_file(fired_gdf_path)

# Load western ecoregions and reproject to match fire dataset CRS
western_eco = gpd.read_file(western_eco_path).to_crs(fired_gdf.crs)

# Quick visualization of ecoregion boundaries for sanity checking spatial alignment
western_eco.plot()

In [ ]:
# Clip fire events to western ecoregion boundaries
fired_clip = fired_gdf.clip(western_eco)

# Save clipped dataset to disk
fired_clip.to_file(clipped_path)

# Reload clipped dataset and parse date fields
fired_clip = gpd.read_file(clipped_path, parse_dates=["ig_date", "last_date"])

# Filter out future/incomplete year entries
fired_clip = fired_clip[fired_clip['ig_year'] < 2025]

In [ ]:
# Self-intersection overlay to identify overlapping fire events (reburns)
overlay = gpd.overlay(fired_clip, fired_clip, how="intersection")

# Keep only ordered pairs (earlier fire vs later fire)
overlay = overlay[overlay['ig_year_1'] < overlay['ig_year_2']]

# Compute reburned area in square kilometers
overlay["reburn_area"] = overlay.geometry.area / 1e6 

# Filter out very small overlap artifacts
overlay = overlay[overlay["reburn_area"] > 1]

# Create unique identifier for each reburn pair
overlay["reburn_id"] = (
    overlay["merge_id_1"].astype(str) + "_" + overlay["merge_id_2"].astype(str)
)

# Save overlay results to disk
overlay.to_file(overlay_path)

# Reproject datasets to analysis CRS for raster operations
large_reburns = overlay.to_crs(epsg=5070)
fired_gdf = fired_gdf.to_crs(epsg=5070)

In [ ]:
# Path to NLCD land cover raster (annual product)
nlcd = os.path.join(data_path, "nlcd", "Annual_NLCD_LndCov_2000_CU_C1V1.tif")

# Open NLCD raster as an xarray dataset
nlcd_xr = xr.open_dataset(nlcd)

# Extract coordinate reference system from raster metadata
nlcd_crs = nlcd_xr.rio.crs

In [ ]:
# Initialize counters and storage for loop processing results
row_i = 0
df_len = len(large_reburns)
outputs = []
diff_skip = 0
skipped_rows = []
valid_reburn = 0

# Iterate over each reburn event (fire-fire intersection pairs)
for i, row in large_reburns.iterrows():

    # Progress logging every 100 rows
    if row_i % 100 == 0:
        print(f"On row {row_i} of {df_len}")
        logger1.log(logging.INFO, f"On row {row_i} of {df_len}")

    if valid_reburn % 100 == 0:
        print(f"Added {valid_reburn} total events.")

    row_i += 1

    # Initialize placeholders for computed outputs
    diff_ave, overlay_ave = 0, 0

    # Extract identifiers and metadata for the fire pair
    reburn_id = row["reburn_id"]
    fire1_id, fire2_id = row["merge_id_1"], row["merge_id_2"]
    fire1_year, fire2_year = row["ig_year_1"], row["ig_year_2"]
    fire1_start, fire1_end = row["ig_date_1"], row["last_date_1"]
    fire2_start, fire2_end = row["ig_date_2"], row["last_date_2"]

    # Create geometries for overlap/difference analysis between fire events
    diff_geom, overlay_geom, fire1_geom, fire2_geom = get_diff_overlay_geom(
        fired_gdf, fire1_id, fire2_id
    )

    # Only proceed if a valid difference geometry exists
    if len(diff_geom.geometry.values) != 0:

        # Compute area of non-overlapping region
        diff_area = cal_area(diff_geom)

        if diff_area > .1:

            # Convert geometries into GeoDataFrames for raster operations
            diff_gdf = gpd.GeoDataFrame(geometry=[diff_geom.iloc[0]], crs=fired_gdf.crs)
            overlay_gdf = gpd.GeoDataFrame(geometry=[overlay_geom.iloc[0]], crs=fired_gdf.crs)

            # Compute overlay area
            overlay_area = cal_area(overlay_geom)

            # Reproject to NLCD CRS for land cover analysis
            overlay_nlcd = overlay_gdf.to_crs(nlcd_crs)
            diff_nlcd = diff_gdf.to_crs(nlcd_crs)

            # Extract raw geometry objects for downstream functions
            diff_geo = diff_gdf.geometry.values[0]
            overlay_geo = overlay_geom.geometry.values[0]

            # Check forest cover for overlay and difference regions
            overlay1_forest, overlay1_perc = check_forest_cover(
                fire1_year, overlay_nlcd.geometry, data_path
            )
            overlay2_forest, overlay2_perc = check_forest_cover(
                fire2_year, overlay_nlcd.geometry, data_path
            )
            diff_forest, diff_perc = check_forest_cover(
                fire2_year, diff_nlcd.geometry, data_path
            )

            # Only keep meaningful forest-dominated reburns
            if overlay1_perc >= .5 or overlay2_perc >= .5:

                valid_reburn += 1

                # Compute CBI metrics for fire and overlap regions
                overlay1_cbi = mean_cbi(overlay_gdf, fire1_year, cbi_da)
                overlay2_cbi = mean_cbi(overlay_gdf, fire2_year, cbi_da)
                fire1_cbi = mean_cbi(fire1_geom, fire1_year, cbi_da)
                fire2_cbi = mean_cbi(fire2_geom, fire2_year, cbi_da)
                diff_cbi = mean_cbi(diff_gdf, fire2_year, cbi_da)

                # Extract climate variables for each region/time period
                fire1_climate_dict = extract_gridmet_means(
                    fire1_geom, fire1_year, fire1_start, fire1_end, data_path
                )
                fire2_climate_dict = extract_gridmet_means(
                    fire2_geom, fire2_year, fire2_start, fire2_end, data_path
                )
                overlay1_climate_dict = extract_gridmet_means(
                    overlay_gdf, fire1_year, fire1_start, fire1_end, data_path
                )
                overlay2_climate_dict = extract_gridmet_means(
                    overlay_gdf, fire2_year, fire2_start, fire2_end, data_path
                )
                diff_climate_dict = extract_gridmet_means(
                    diff_gdf, fire2_year, fire2_start, fire2_end, data_path
                )

                # Assign ecoregion classification
                ecoregion_code = get_ecoregion(western_epa, overlay_gdf)

                # Store all computed features in a structured dataframe
                outputs.append(
                    pd.DataFrame(
                        {
                            "fire1_id": fire1_id,
                            "fire2_id": fire2_id,
                            "year_1": fire1_year,
                            "year_2": fire2_year,
                            "fire1_start": fire1_start,
                            "fire1_end": fire1_end,
                            "fire2_start": fire2_start,
                            "fire2_end": fire2_end,

                            # CBI variables
                            "fire1_cbi": fire1_cbi,
                            "fire2_cbi": fire2_cbi,
                            "reburn1_cbi": overlay1_cbi,
                            "reburn2_cbi": overlay2_cbi,
                            "non_reburn_cbi": diff_cbi,

                            # Climate variables — fire1
                            "fire1_pr": fire1_climate_dict["pr"],
                            "fire1_tmmn": fire1_climate_dict["tmmn"],
                            "fire1_tmmx": fire1_climate_dict["tmmx"],
                            "fire1_vpd": fire1_climate_dict["vpd"],

                            # Climate variables — fire2
                            "fire2_pr": fire2_climate_dict["pr"],
                            "fire2_tmmn": fire2_climate_dict["tmmn"],
                            "fire2_tmmx": fire2_climate_dict["tmmx"],
                            "fire2_vpd": fire2_climate_dict["vpd"],

                            # Climate variables — overlay (fire1 year)
                            "reburn1_pr": overlay1_climate_dict["pr"],
                            "reburn1_tmmn": overlay1_climate_dict["tmmn"],
                            "reburn1_tmmx": overlay1_climate_dict["tmmx"],
                            "reburn1_vpd": overlay1_climate_dict["vpd"],

                            # Climate variables — overlay (fire2 year)
                            "reburn2_pr": overlay2_climate_dict["pr"],
                            "reburn2_tmmn": overlay2_climate_dict["tmmn"],
                            "reburn2_tmmx": overlay2_climate_dict["tmmx"],
                            "reburn2_vpd": overlay2_climate_dict["vpd"],

                            # Climate variables — non-reburn region
                            "non_reburn_pr": diff_climate_dict["pr"],
                            "non_reburn_tmmn": diff_climate_dict["tmmn"],
                            "non_reburn_tmmx": diff_climate_dict["tmmx"],
                            "non_reburn_vpd": diff_climate_dict["vpd"],

                            # Forest cover metrics
                            "reburn1_forest": overlay1_forest,
                            "reburn2_forest": overlay2_forest,
                            "non_reburn_forest": diff_forest,
                            "reburn1_perc": overlay1_perc,
                            "reburn2_perc": overlay2_perc,
                            "non_reburn_perc": diff_perc,

                            # Ecoregion and spatial summary stats
                            "ecoregion": ecoregion_code,
                            "year_gap": fire2_year - fire1_year,
                            "diff_area": diff_area,
                            "reburn_area": overlay_area,

                            # Geometry output for final dataset
                            "geometry": overlay_geo
                        },
                        index=[1],
                    )
                )

        else:
            # Track skipped cases where difference geometry is too small or invalid
            diff_skip += 1
            skipped_rows.append(reburn_id)

    else:
        # Track cases with missing difference geometry
        diff_skip += 1
        skipped_rows.append(reburn_id)

        if diff_skip % 10 == 0:
            print(f"Skipped {diff_skip} rows due to no difference geometry")

In [ ]:
# Final stage: consolidate results and write output dataset
print("Outputting files!")

# Combine all per-reburn results into a single DataFrame
main_gdf = pd.concat(outputs)

# Convert to GeoDataFrame with geographic CRS
final_gdf = gpd.GeoDataFrame(
    main_gdf,
    geometry=main_gdf.geometry,
    crs="EPSG:4326"
)

# Remove duplicate rows if any were generated during processing
final_gdf = final_gdf.drop_duplicates()

# Write final reburn dataset to disk
final_gdf.to_file(final_path)